# Local Vision Annotator demo notebook

This notebook demonstrates the local annotation workflow using the sample images in `data/`. It uses the same project structure as the Streamlit app, so annotations created here can also be opened in the app, exported to YOLO, or continued later.

The sample task is intentionally simple: annotate visible cardboard boxes as `BOX`. You can replace the project name, image directory, classes, and instructions to reuse the notebook for any object detection task.

## 1. Configure the demo project

The default configuration points to `PROJECT_ROOT / "data"`, which contains three sample warehouse images. Running this cell creates or updates `annotations/demo_boxes/` and indexes the images.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from annotation_app.project_io import create_or_update_project, import_legacy_yolo_labels, load_image_index, progress
from annotation_app.notebook_annotator import run_notebook_annotator
from annotation_app.exporter import export_yolo

PROJECTS_ROOT = PROJECT_ROOT / 'annotations'
PROJECTS_ROOT.mkdir(exist_ok=True)

PROJECT_NAME = 'demo_boxes'
IMAGE_DIR = PROJECT_ROOT / 'data'
CLASSES = [
    {'id': 0, 'name': 'BOX', 'color': '#f59e0b'},
]
INSTRUCTIONS = (
    'Annotate each visible cardboard box. Include partially visible boxes when the box boundary is useful. '
    'Ignore pallets, forklifts, floor markings, plastic wrap, and background shelves.'
)

project_path = create_or_update_project(PROJECTS_ROOT, PROJECT_NAME, str(IMAGE_DIR), CLASSES, INSTRUCTIONS)
images = load_image_index(project_path)

print(f'Project path: {project_path}')
print(f'Image directory: {IMAGE_DIR}')
print(f'Indexed images: {len(images)}')
for item in images:
    print(f"- {item['relative_path']}")

Project path: d:\Projetos\local-vision-annotator\annotations\demo_boxes
Image directory: d:\Projetos\local-vision-annotator\data
Indexed images: 3
- image_01.jpeg
- image_02.jpeg
- image_03.jpeg


## 2. Annotate from the notebook

This opens an OpenCV window. Use the mouse to draw boxes around the sample boxes.

Controls:

- Mouse: draw bounding box
- `ENTER`: save as annotated and move forward
- `D`: mark as empty
- `R`: mark for review
- `S`: skip
- `Z`: undo last box
- `C`: cycle class
- `0-9`: select class by id
- `N` / `P`: next / previous image
- `ESC`: exit

In [2]:
run_notebook_annotator(
    project_path,
    status_filter=('pending', 'needs_review'),
    reannotate=False,
    # Increase these if your monitor has room.
    max_width=1800,
    max_height=1200,
)

{'pending': 0, 'annotated': 3, 'empty': 0, 'skipped': 0, 'needs_review': 0}


## 3. Check progress

The notebook and Streamlit app both read the same metadata, so this summary reflects work done in either interface.

In [3]:
images = load_image_index(project_path)
print(progress(project_path, images))

{'pending': 0, 'annotated': 3, 'empty': 0, 'skipped': 0, 'needs_review': 0, 'total': 3}


## 4. Optional: import existing YOLO labels

Use this only when you already have labels in a sibling label folder. The importer matches labels to images by filename stem.

In [ ]:
LEGACY_LABELS_DIR = IMAGE_DIR / 'labels'
LEGACY_CLASS_ID = 0

if LEGACY_LABELS_DIR.exists():
    result = import_legacy_yolo_labels(project_path, LEGACY_LABELS_DIR, class_id=LEGACY_CLASS_ID)
    print(result)
else:
    print(f'No legacy labels found at: {LEGACY_LABELS_DIR}')

## 5. Export a YOLO dataset

After annotating at least one image, export a YOLO dataset with reproducible train/validation/test splits. Empty images can be included as negative examples.

In [4]:
export_dir = export_yolo(
    project_path,
    train_pct=80,
    val_pct=15,
    seed=42,
    include_empty=True,
)
print(f'Dataset exported to: {export_dir}')

Dataset exported to: d:\Projetos\local-vision-annotator\annotations\demo_boxes\exports\yolo_2026_06_17_170148


## 6. Open the same project in Streamlit

The browser app can continue the same `demo_boxes` project:

```bash
streamlit run annotation_app/app.py
```

Choose `demo_boxes` in the sidebar.